# Bab 13. Aljabar Linear dan Turunan Numerik

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## Persiapan

Bab ini melanjutkan contoh dari bab sebelumnya. Jalankan sel ini lebih dahulu supaya datanya tersedia.

In [ ]:
import numpy as np
from siapkan import matriks_desain

Xb, y = matriks_desain()
n = len(y)

## 1. Menghitung norma

In [ ]:
import numpy as np

v = np.array([3., -4., 12.])
print(np.linalg.norm(v, 1))
print(np.linalg.norm(v))          # bawaan: L2
print(np.linalg.norm(v, np.inf))

Keluaran yang diharapkan:

```
19.0
13.0
12.0
```

## 2. Membaca sebuah matriks

In [ ]:
A = np.array([[3., 1.],
              [1., 2.]])

U, S, Vt = np.linalg.svd(A)
print(np.round(S, 6))
print(round(np.linalg.det(A), 6))
print(round(S[0] / S[-1], 6))

Keluaran yang diharapkan:

```
[3.618034 1.381966]
5.0
2.618034
```

## 3. Menguji aturan praktis

In [ ]:
for n in [4, 8, 12]:
    H = np.array([[1/(i+j+1) for j in range(n)]
                  for i in range(n)])
    x = np.ones(n)
    b = H @ x
    xh = np.linalg.solve(H, b)
    galat = np.linalg.norm(xh - x) / np.linalg.norm(x)
    print(f"{np.linalg.cond(H):.2e} {galat:.2e}")

Keluaran yang diharapkan:

```
1.55e+04 3.37e-14
1.53e+10 3.71e-08
1.64e+16 1.29e-01
```

## 4. Persamaan normal versus lstsq

In [ ]:
for delta in [1e-5, 1e-8]:
    A = np.array([[1., 1.], [delta, 0.], [0., delta]])
    b = np.array([2., delta, delta])      # jawaban: (1, 1)

    try:
        w = np.linalg.solve(A.T @ A, A.T @ b)
        print("normal:", np.linalg.norm(w - 1))
    except np.linalg.LinAlgError as e:
        print("normal: GAGAL:", e)

    w = np.linalg.lstsq(A, b, rcond=None)[0]
    print("lstsq :", np.linalg.norm(w - 1))

Keluaran yang diharapkan:

```
normal: 7.07e-11
lstsq : 3.14e-16
normal: GAGAL: Singular matrix
lstsq : 4.97e-16
```

## 5. Memeriksa permukaan biaya

In [ ]:
H = (2/n) * Xb.T @ Xb        # Xb: matriks desain Bab 13
print(np.round(np.linalg.eigvalsh(H), 4))
print(round(np.linalg.cond(Xb), 4))
print(np.round(np.linalg.svd(Xb)[1], 4))

Keluaran yang diharapkan:

```
[0.1032 2.     3.8968]
6.1438
[3.948  2.8284 0.6426]
```

## 6. Mencari langkah terbaik

In [ ]:
f, fp, x0 = np.sin, np.cos, 1.0
sejati = fp(x0)

hs = np.logspace(-1, -16, 61)
maju = np.abs((f(x0+hs) - f(x0))/hs - sejati)
pusat = np.abs((f(x0+hs) - f(x0-hs))/(2*hs) - sejati)

print(hs[maju.argmin()], maju.min())
print(hs[pusat.argmin()], pusat.min())

Keluaran yang diharapkan:

```
5.62e-09 2.53e-09
5.62e-06 3.76e-13
```

## 7. Pemeriksaan gradien

In [ ]:
def grad_numerik(f, w, h=1e-6):
    g = np.zeros_like(w)
    for i in range(len(w)):
        wp, wm = w.copy(), w.copy()
        wp[i] += h
        wm[i] -= h
        g[i] = (f(wp) - f(wm)) / (2*h)
    return g

def J(w):
    r = Xb @ w - y
    return r @ r / n

def grad_analitik(w):
    return (2/n) * Xb.T @ (Xb @ w - y)

rng = np.random.default_rng(0)
w = rng.normal(0, 5, Xb.shape[1])
ga, gn = grad_analitik(w), grad_numerik(J, w)
print(np.linalg.norm(ga - gn) /
      (np.linalg.norm(ga) + np.linalg.norm(gn)))

Keluaran yang diharapkan:

```
1.542e-08
```

## 8. Menangkap gradien yang salah

In [ ]:
def grad_salah(w):
    return (1/n) * Xb.T @ (Xb @ w - y)   # faktor 2 hilang

gs = grad_salah(w)
print(np.linalg.norm(gs - gn) /
      (np.linalg.norm(gs) + np.linalg.norm(gn)))

Keluaran yang diharapkan:

```
3.333e-01
```